# 06 - Ejercicios Practicos: Monitoreo Transaccional SWIFT y Deteccion AML con DBSCAN

## Curso de Machine Learning: Modelos No Supervisados

En este cuaderno pondras en practica el flujo completo de **Agrupamiento Basado en Densidad con DBSCAN** aplicado a uno de los casos de uso financieros mas demandados en la banca internacional y las FinTech: la **Segmentacion de Perfiles Tesoreros Corporativos y Deteccion No Supervisada de Lavado de Activos (AML / PLD) en Transferencias Transfronterizas SWIFT**.

> **Nota:** Encontraras las soluciones completas y detalladas a estos ejercicios en la carpeta `.soluciones/06_Ejercicios_Practicos_Solucion.ipynb`. Te recomendamos resolver cada ejercicio por tu cuenta antes de consultar la solucion.

---

### Instrucciones Generales
1. Ejecuta la celda de configuracion inicial y generacion del dataset bancario SWIFT.
2. Lee cuidadosamente el contexto financiero y los pasos solicitados en cada ejercicio.
3. Escribe tu codigo en las celdas marcadas con `# TU CODIGO AQUI`.
4. Redacta tus interpretaciones financieras en las celdas de Markdown correspondientes.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## Contexto del Dataset Financiero: Auditoria de Transferencias Corporativas SWIFT y AML (650 Empresas)

La Unidad de Inteligencia Financiera y Cumplimiento (*Compliance AML*) de un banco global analiza el comportamiento anual de **650 cuentas corporativas** que operan en la red de pagos internacionales SWIFT, descritas mediante **7 variables financieras de comportamiento transaccional**:

- `rotacion_fondos_mensual`: Velocidad con la que el dinero entra y sale de la cuenta en el mismo mes (veces/mes). Valores extremos sugieren cuentas puente (*pass-through / layering*).
- `ticket_promedio_miles_usd`: Monto promedio por transferencia internacional enviada (en miles de USD).
- `pct_flujo_offshore`: Porcentaje del volumen anual enviado hacia jurisdicciones catalogadas como paraisos fiscales (%).
- `entropia_horaria_nocturna`: Indice de actividad fuera de horario bancario estandar (0 a 10).
- `indice_concentracion_hhi`: Concentracion de contrapartes beneficiarias (0 a 100).
- `desviacion_ley_benford_chi2`: Estadistico $\chi^2$ que mide cuanto se desvian los primeros digitos de los montos facturados respecto a la Ley de Benford (valores altos indican montos fraccionados artificialmente o *smurfing*).
- `spread_divisas_exoticas_bps`: Sobrecosto medio asumido en conversion de divisas iliquidas (puntos basicos, bps).


In [ ]:
# Generador reproducible de perfiles transaccionales corporativos SWIFT + Anomalias AML (n=650)
def cargar_dataset_swift_aml(seed=42):
    rng = np.random.default_rng(seed)

    # Regimen 1: Importadores / Exportadores PyME Tradicionales (n=280)
    c1 = np.column_stack([
        rng.normal(2.2, 0.5, 280).clip(0.8, 4.0),     # rotacion_fondos_mensual
        rng.normal(45.0, 12.0, 280).clip(15.0, 85.0), # ticket_promedio_miles_usd
        rng.normal(4.5, 2.0, 280).clip(0.0, 12.0),    # pct_flujo_offshore
        rng.normal(1.8, 0.5, 280).clip(0.2, 3.5),     # entropia_horaria_nocturna
        rng.normal(62.0, 9.0, 280).clip(35.0, 88.0),  # indice_concentracion_hhi
        rng.normal(3.2, 1.1, 280).clip(0.5, 7.0),     # desviacion_ley_benford_chi2
        rng.normal(18.0, 4.0, 280).clip(8.0, 30.0)    # spread_divisas_exoticas_bps
    ])

    # Regimen 2: Tesorerias de Corporativos Multinacionales (Alto ticket, alta concentracion intragrupo) (n=210)
    c2 = np.column_stack([
        rng.normal(4.8, 0.8, 210).clip(2.5, 7.5),       # rotacion_fondos_mensual
        rng.normal(850.0, 140.0, 210).clip(450.0, 1300.0), # ticket_promedio_miles_usd
        rng.normal(18.0, 4.5, 210).clip(6.0, 32.0),     # pct_flujo_offshore
        rng.normal(4.2, 0.8, 210).clip(2.0, 6.5),       # entropia_horaria_nocturna
        rng.normal(82.0, 6.0, 210).clip(65.0, 96.0),    # indice_concentracion_hhi
        rng.normal(2.8, 0.9, 210).clip(0.5, 6.0),       # desviacion_ley_benford_chi2
        rng.normal(9.5, 2.5, 210).clip(3.0, 18.0)       # spread_divisas_exoticas_bps
    ])

    # Regimen 3: Plataformas FinTech / Remesadoras yAgregadores de Pago (Alta rotacion, bajo ticket, baja concentracion) (n=125)
    c3 = np.column_stack([
        rng.normal(14.5, 2.0, 125).clip(9.5, 20.0),    # rotacion_fondos_mensual
        rng.normal(12.0, 3.5, 125).clip(3.0, 24.0),    # ticket_promedio_miles_usd
        rng.normal(11.0, 3.0, 125).clip(3.0, 20.0),    # pct_flujo_offshore
        rng.normal(7.5, 0.9, 125).clip(5.2, 9.2),      # entropia_horaria_nocturna
        rng.normal(18.0, 4.5, 125).clip(6.0, 32.0),    # indice_concentracion_hhi
        rng.normal(4.0, 1.2, 125).clip(1.0, 7.5),      # desviacion_ley_benford_chi2
        rng.normal(42.0, 6.5, 125).clip(25.0, 60.0)    # spread_divisas_exoticas_bps
    ])

    # Anomalias Forenses AML (Cuentas cascaron / Smurfing / Layering Offshore) (n=35)
    aml = np.column_stack([
        rng.uniform(28.0, 65.0, 35),     # rotacion_fondos_mensual extrema
        rng.uniform(1900.0, 4800.0, 35), # ticket_promedio_miles_usd atipico
        rng.uniform(62.0, 98.0, 35),     # pct_flujo_offshore paraisos fiscales
        rng.uniform(8.5, 10.0, 35),      # entropia_horaria_nocturna
        rng.uniform(5.0, 95.0, 35),      # indice_concentracion_hhi erratico
        rng.uniform(22.0, 58.0, 35),     # desviacion_ley_benford_chi2 (violacion Ley de Benford)
        rng.uniform(110.0, 260.0, 35)    # spread_divisas_exoticas_bps
    ])

    X_mat = np.vstack([c1, c2, c3, aml])
    perm = rng.permutation(len(X_mat))
    columnas = [
        'rotacion_fondos_mensual', 'ticket_promedio_miles_usd', 'pct_flujo_offshore',
        'entropia_horaria_nocturna', 'indice_concentracion_hhi',
        'desviacion_ley_benford_chi2', 'spread_divisas_exoticas_bps'
    ]
    cuentas_id = [f'SWIFT_CORP_{i:03d}' for i in range(1, len(X_mat) + 1)]
    return pd.DataFrame(X_mat[perm], index=cuentas_id, columns=columnas)

df_swift = cargar_dataset_swift_aml()
print(f"Dimensiones del dataset bancario SWIFT: {df_swift.shape}")
df_swift.head()


---
## Ejercicio 1: Preprocesamiento Financiero Robusto ante Anomalias

1. Imprime el resumen estadistico de `df_swift` incluyendo los percentiles `50%` (mediana), `75%` y `max`.
2. Transforma el dataset utilizando `RobustScaler` y guarda el resultado en `X_rob`.
3. Transforma tambien el dataset con `StandardScaler` en `X_std` y compara el rango intercuartilico de las variables normales en ambas matrices.
4. **Pregunta:** ¿Que variables presentan la mayor brecha entre el percentil 75% y el valor maximo, sugiriendo la presencia de actividad financiera anomala?


In [ ]:
# TU CODIGO AQUI: Estadisticas descriptivas y escalado con RobustScaler



**Tus Observaciones (Ejercicio 1):**
*Escribe aqui tu respuesta...*


---
## Ejercicio 2: Calibracion Matematica de $\varepsilon$ con el Grafico de $k$-Distancias

Dado que nuestro dataset tiene $p = 7$ variables financieras, utilizaremos la regla de Sander fijando `min_samples = 14` ($2 \times p$).
1. Ajusta un modelo `NearestNeighbors(n_neighbors=14)` sobre la matriz `X_rob`.
2. Extrae las distancias al 14-esimo vecino mas cercano, ordenalas de menor a mayor y graficalas.
3. Identifica visual o matematicamente el valor de $\varepsilon^*$ en la rodilla de la curva donde inicia la pared vertical de anomalias AML.


In [ ]:
# TU CODIGO AQUI: Curva de k-distancias con NearestNeighbors(n_neighbors=14)



**Tus Observaciones (Ejercicio 2):**
*Escribe aqui el rango optimo de `eps` que observas en la rodilla del grafico...*


---
## Ejercicio 3: Busqueda en Cuadricula y Evaluacion Limpia de Ruido

1. Evalua una cuadricula de combinaciones con `eps` $\in [0.6, 0.8, 1.0, 1.2, 1.5]$ y `min_samples` $\in [10, 14, 18]$.
2. Para cada combinacion, calcula:
   - Numero de regimenes descubiertos (excluyendo `-1`).
   - Porcentaje de cuentas marcadas como alerta AML (`label == -1`).
   - **Silhouette Score Limpio** (evaluado exclusivamente sobre las muestras con `label != -1`).
3. Selecciona la configuracion optima que descubra los regimenes legitimos con alto Silhouette y mantenga una tasa de alertas AML razonable (entre 4% y 7% del portafolio).


In [ ]:
# TU CODIGO AQUI: Grid Search personalizado para DBSCAN



---
## Ejercicio 4: Perfilamiento de Regimenes Tesoreros y Auditoria de Alertas AML (`-1`)

1. Entrena el modelo DBSCAN optimo (`eps=1.0, min_samples=14`) sobre `X_rob` y guarda las etiquetas en `df_swift['Cluster']`.
2. Calcula la tabla de **medianas en unidades financieras originales** agrupando por `'Cluster'` (incluyendo el grupo `-1`).
3. Genera un mapa de calor (`sns.heatmap`) de las medianas en el espacio `RobustScaler` y un grafico de dispersion en 2D proyectado con `PCA(n_components=2)` destacando las alertas `-1` con marcadores rojos `'x'`.


In [ ]:
# TU CODIGO AQUI: Tabla de medianas, Heatmap y Proyeccion PCA



---
## Ejercicio 5: Reglas de Cumplimiento Bancario mediante Arbol Sustituto

1. Entrena un `DecisionTreeClassifier(max_depth=3, random_state=42)` utilizando las 7 variables originales de `df_swift` como entrada y `df_swift['Cluster']` como salida.
2. Imprime la fidelidad (*accuracy*) del arbol, muestra sus reglas en texto (`export_text`) y graficalo con `plot_tree`.
3. **Pregunta Final:** Segun el arbol de decision y la tabla de medianas, ¿que umbrales en `desviacion_ley_benford_chi2`, `pct_flujo_offshore` o `rotacion_fondos_mensual` caracterizan a las cuentas sospechosas de lavado de activos (`Cluster -1`)?


In [ ]:
# TU CODIGO AQUI: Arbol de decision sustituto para el Comite de Cumplimiento AML



**Tus Conclusiones y Reglas Forenses AML (Ejercicio 5):**
- **Regimen 0:** *...*
- **Regimen 1:** *...*
- **Regimen 2:** *...*
- **Perfil de Alerta Forense AML (Cluster -1):** *...*
